# DAY 159 - Fine-tuning GPT2.
@A.IPYNB

# Fine-Tuning GPT-2: The "Hello World" of LLM Training

### The Concept: Causal Language Modeling (CLM)
GPT (Generative Pre-trained Transformer) models are trained on one simple task: **Predict the next token.**
$$P(w_t | w_{1}, w_{2}, ..., w_{t-1})$$

### The Objective
We will fine-tune `gpt2` (124M parameters) on a dataset of **Inspirational Quotes**.
* **Before Training:** It will ramble like a generic internet user.
* **After Training:** It will generate profound (or pseudo-profound) philosophical statements.

### Why GPT-2 in 2026?
While Llama 3 and Mistral are the SOTA, GPT-2 remains the perfect laboratory.
1.  **Fast:** Trains in minutes on a free Colab GPU.
2.  **Transparent:** You can see every layer and attention head easily.
3.  **Foundational:** The architecture (Decoder-only Transformer) is identical to modern giants.

In [1]:
# @title 1. Dependencies
# We need 'transformers' for the model, 'datasets' for data handling,
# and 'accelerate' for efficient training loop.
!pip install -q transformers datasets accelerate

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [2]:
# @title 2. Load and Prepare Data
# We will use a dataset of English quotes to teach the model a "philosophical" style.
dataset = load_dataset("Abirate/english_quotes")

# Split into train/test
train_dataset = dataset["train"].train_test_split(test_size=0.1)

print(f"Training Samples: {len(train_dataset['train'])}")
print(f"Sample: {train_dataset['train'][0]['quote']}")
print(f"Author: {train_dataset['train'][0]['author']}")

README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Training Samples: 2257
Sample: “When I saw you I fell in love, and you smiled because you knew.”
Author: Arrigo Boito


### The Critical Step: EOS Token
In Causal Language Modeling, the model needs to know when a sentence ends.
We must append the `EOS` (End of String) token to every example.

**Format:** `Quote + <|endoftext|>`

In [3]:
# @title 3. Tokenize the Data
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 doesn't have a padding token by default, so we use EOS
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # HERE I AM Adding EOS token to the end of every quote so the model learns to stop generating
    inputs = [quote + tokenizer.eos_token for quote in examples["quote"]]
    return tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

tokenized_datasets = train_dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["quote", "author", "tags"])
tokenized_datasets.set_format("torch")

print("Data Tokenized.")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/2257 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

Data Tokenized.


In [4]:
# @title 4. Initialize GPT-2
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

# DO We need a Data Collator that handles "Masking" for us?
# NO! For GPT-2 (CLM), we don't mask. We shift labels.
# The DataCollatorForLanguageModeling(mlm=False) handles this automatically.
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("Model loaded. Ready to learn wisdom.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded. Ready to learn wisdom.


In [20]:
# @title 5. Model
training_args = TrainingArguments(
    output_dir="./gpt2-quotes",
    num_train_epochs=3,              # 3 loops over the data
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
50,2.798973
100,2.737699
150,2.833353
200,2.815386
250,2.887355
300,2.821432
350,2.701818
400,2.698258
450,2.675943
500,2.723762


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=849, training_loss=2.741328514647287, metrics={'train_runtime': 263.1847, 'train_samples_per_second': 25.727, 'train_steps_per_second': 3.226, 'total_flos': 442302087168000.0, 'train_loss': 2.741328514647287, 'epoch': 3.0})

### Visualizing the Change
We will now ask the model to generate text starting with "The meaning of life is".

**Temperature:** Controls randomness.
* Low (0.2): Boring, repetitive, safe.
* High (1.0): Creative, chaotic, potentially nonsensical.

In [38]:
def generate_quote_optimized(prompt, max_length=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=3,                    # Beam search for coherence
        num_return_sequences=1,
        early_stopping=True,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=2,
    )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated = generated[len(prompt):].strip()
    generated = clean_quote(generated)

    return generated

def clean_quote(text):
    if not text:
        return text
    text = ' '.join(text.split())

    sentences = []
    for sentence in text.split('. '):
        sentence = sentence.strip()
        if not sentence:
            continue

        if sentence and sentence[0].islower():
            sentence = sentence[0].upper() + sentence[1:]

        if sentence.endswith('."'):
            sentence = sentence[:-2] + '".'
        elif sentence.endswith('!.'):
            sentence = sentence[:-2] + '!'
        elif sentence.endswith('?.'):
            sentence = sentence[:-2] + '?'

        if sentence and sentence[-1] not in '.!?"\'':
            sentence += '.'

        sentences.append(sentence)

    if sentences:
        return sentences[0]
    return text

# Prompt List
prompts = [
    "True happiness comes from",
    "Life resembles",
    "Wisdom teaches us that",
    "The meaning of life is",
    "Love is",
    "Success means",
]

print("- Quotes -")
for i, prompt in enumerate(prompts[:6]):  # Try first 3
    quote = generate_quote_optimized(prompt)
    print(f"{prompt} {quote}")
    print()

- Quotes -
True happiness comes from Loving people, not from hating them.”If you want to be happy, you have to love people as much as you love yourself, and that means loving yourself more than you can love someone else.

Life resembles A book.

Wisdom teaches us that The only way to happiness is to be a good person.

The meaning of life is Not what it looks like, but what you can see and feel.

Love is The most beautiful thing in the world.”It's the only thing that can make you feel loved, and it's only when you're in love with someone that you realize that love is real, that it really does exist.

Success means That you can't be happy without suffering.



### The Strategic Win

We have successfully fine-tuned a Generative Pre-trained Transformer.

1.  **Architecture:** We used the standard Decoder-only stack.
2.  **Data:** We formatted text for Causal Language Modeling (CLM).
3.  **Result:** We altered the model's weights to bias it toward a specific style (Quotes).

**From here to Llama 3:**
The code We just wrote is 90% identical to what We would use to fine-tune Llama 3 8B. The only difference is the model size and the need for LoRA (which we have covered in the T5 lesson) to fit it in memory.